In [1]:
from google.colab import files
uploaded = files.upload()

Saving sql_query.csv to sql_query.csv
Saving sql_query3.csv to sql_query3.csv


In [2]:
# ── STEP 1: Load & merge datasets ───────────────────────────────
import pandas as pd
import numpy as np
import re, pickle

df1 = pd.read_csv("sql_query.csv")
df2 = pd.read_csv("sql_query3.csv")
df  = pd.concat([df1, df2], ignore_index=True)

In [3]:
df['query'] = df['query'].str.lower().str.strip()
df = df.dropna().drop_duplicates()
print("Total training samples:", len(df))
print("Unique intents:", sorted(df['intent'].unique()))

Total training samples: 611
Unique intents: ['all_announcement', 'all_attendance', 'all_attendance_number', 'all_leave', 'all_leave_balance', 'all_status', 'all_status_number', 'all_today_announcement', 'all_today_attendance', 'all_today_leave', 'all_today_status', 'all_today_wfh', 'all_tomorrow_announcement', 'all_tomorrow_attendance', 'all_tomorrow_leave', 'all_tomorrow_status', 'all_tomorrow_wfh', 'all_wfh', 'all_yesterday_announcement', 'all_yesterday_attendance', 'all_yesterday_leave', 'all_yesterday_status', 'all_yesterday_wfh', 'announcement', 'approved_wfh', 'attendance', 'attendance_report_last_n_days', 'attendance_today', 'leave', 'leave_approved', 'leave_balance', 'leave_history', 'leave_pending', 'leave_rejected', 'leave_status', 'pending_leave', 'pending_wfh', 'rejected_wfh', 'status', 'status_number', 'today_announcement', 'today_attendance', 'today_leave', 'today_status', 'today_wfh', 'tomorrow_announcement', 'tomorrow_attendance', 'tomorrow_leave', 'tomorrow_status', 't

In [4]:
# ── STEP 3: Normalize — replace employee names with <PERSON> ────
#  This is CRITICAL: the trained model must see <PERSON> at inference too.
#  The chatbot_engine.py normalize_text() does the same thing at runtime.

KEEP_WORDS = {
    'leave','balance','show','wfh','status','attendance','pending',
    'today','yesterday','tomorrow','all','employees','announcements',
    'give','me','the','of','is','what','work','from','home','days',
    'provide','details','requests','number','a','an','for'
}

def normalize(text):
    text = str(text).lower().strip()
    tokens = text.split()
    result = []
    for t in tokens:
        clean = re.sub(r'[^a-z]', '', t)
        if clean and clean not in KEEP_WORDS and len(clean) > 2:
            result.append('<PERSON>')
        else:
            result.append(t)
    return ' '.join(result)

df['query_norm'] = df['query'].apply(normalize)

In [6]:
# ── STEP 4: Train / Test split ──────────────────────────────────
from sklearn.model_selection import train_test_split

X = df['query_norm']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)}  |  Test: {len(X_test)}")


Train: 488  |  Test: 123


In [7]:
# ── STEP 5: Vectorize ───────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True      # improves performance on imbalanced classes
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

In [8]:
# ── STEP 6: Train model — LogisticRegression (best for this task) ──
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000, C=5, solver='lbfgs', multi_class='multinomial')
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Accuracy: 0.6178861788617886

Classification Report:

                               precision    recall  f1-score   support

             all_announcement       1.00      0.67      0.80         3
        all_attendance_number       1.00      1.00      1.00         5
            all_status_number       0.00      0.00      0.00         2
             all_today_status       1.00      0.80      0.89         5
                all_today_wfh       1.00      0.43      0.60         7
      all_tomorrow_attendance       0.00      0.00      0.00         1
           all_tomorrow_leave       0.50      1.00      0.67         1
          all_tomorrow_status       0.00      0.00      0.00         1
             all_tomorrow_wfh       0.67      1.00      0.80         6
                      all_wfh       0.27      0.60      0.38         5
   all_yesterday_announcement       1.00      1.00      1.00         6
     all_yesterday_attendance       0.00      0.00      0.00         1
          all_yesterd

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
# ── STEP 7: Quick sanity test ───────────────────────────────────
def predict(text):
    norm = normalize(text)
    vec  = vectorizer.transform([norm])
    return model.predict(vec)[0]

test_queries = [
    ("Show leave balance of Ravi",                  "leave_balance"),
    ("Show pending WFH requests of Priya",          "pending_wfh"),
    ("Show pending leave requests of Kumar",        "pending_leave"),
    ("Give me 7 days status of Anjali",             "status_number"),
    ("What is today attendance of Suresh",          "today_attendance"),
    ("Show today status of all employees",          "all_today_status"),
    ("Show leave balance of all employees",         "all_leave_balance"),
    ("Provide 10 days attendance of all employees","all_attendance_number"),
    ("Show today announcements",                    "today_announcement"),
    ("What is yesterday leave status of Ravi",      "yesterday_leave"),
    ("What is tomorrow WFH status of Priya",        "tomorrow_wfh"),
    ("Show attendance of all employees",            "all_attendance"),
]

print("\n── Sanity tests ──")
all_pass = True
for q, expected in test_queries:
    got = predict(q)
    status = "✅" if got == expected else "❌"
    if got != expected:
        all_pass = False
    print(f"  {status}  '{q}'\n      expected={expected}  got={got}")

print("\nAll tests passed!" if all_pass else "\nSome tests failed — check training data.")


── Sanity tests ──
  ✅  'Show leave balance of Ravi'
      expected=leave_balance  got=leave_balance
  ✅  'Show pending WFH requests of Priya'
      expected=pending_wfh  got=pending_wfh
  ✅  'Show pending leave requests of Kumar'
      expected=pending_leave  got=pending_leave
  ✅  'Give me 7 days status of Anjali'
      expected=status_number  got=status_number
  ✅  'What is today attendance of Suresh'
      expected=today_attendance  got=today_attendance
  ✅  'Show today status of all employees'
      expected=all_today_status  got=all_today_status
  ❌  'Show leave balance of all employees'
      expected=all_leave_balance  got=leave_balance
  ✅  'Provide 10 days attendance of all employees'
      expected=all_attendance_number  got=all_attendance_number
  ✅  'Show today announcements'
      expected=today_announcement  got=today_announcement
  ❌  'What is yesterday leave status of Ravi'
      expected=yesterday_leave  got=leave
  ❌  'What is tomorrow WFH status of Priya'
      exp

In [12]:
# ── STEP 8: Save models ─────────────────────────────────────────
pickle.dump(model,      open("chatbot_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl",    "wb"))
print("\nModels saved: chatbot_model.pkl  vectorizer.pkl")


Models saved: chatbot_model.pkl  vectorizer.pkl


In [ ]:
# ── STEP 9: Download ────────────────────────────────────────────
files.download("chatbot_model.pkl")
files.download("vectorizer.pkl")
print("Download complete. Place both files in your Django app folder (same as chatbot_engine.py).")